In [1]:
from datetime import datetime,timedelta
from tldhamburg.backend.Things import Things,DBMaintainer,DataFetcher
import sqlite3
import requests
import time
import json
import os
from concurrent.futures import ThreadPoolExecutor

query = "SELECT thingID, primary_signal from DATASTREAMS"
with sqlite3.connect(DBMaintainer.get_db_path()) as connection:
    cursor = connection.cursor()
    cursor.execute("PRAGMA foreign_keys = ON;")
    cursor.execute(query)
    rows = cursor.fetchall()
    thing_ids = [ID[0] for ID in rows]
    datastream_ids = [ID[1] for ID in rows]


ModuleNotFoundError: No module named 'tldhamburg'

In [5]:
print(len(datastream_ids))

20021


In [ ]:
# Function to fetch data
def fetch_data(url):
    try:
        response = requests.get(url)
        response.raise_for_status()
        return response.json()  # Return parsed JSON
    except requests.RequestException as e:
        return {"error": str(e)}
    
combined_observations = dict()
start_idx = 20020 #last collected index 
counter = 0
for ID in datastream_ids[start_idx:]:
    start = time.time()
    urls = [
        f"https://tld.iot.hamburg.de/v1.1/Datastreams({ID})/Observations?$skip=0&$top=1000&$orderby=phenomenonTime asc",
        f"https://tld.iot.hamburg.de/v1.1/Datastreams({ID})/Observations?$skip=1000&$top=1000&$orderby=phenomenonTime asc",
        f"https://tld.iot.hamburg.de/v1.1/Datastreams({ID})/Observations?$skip=2000&$top=1000&$orderby=phenomenonTime asc",
        f"https://tld.iot.hamburg.de/v1.1/Datastreams({ID})/Observations?$skip=3000&$top=1000&$orderby=phenomenonTime asc",
        f"https://tld.iot.hamburg.de/v1.1/Datastreams({ID})/Observations?$skip=4000&$top=1000&$orderby=phenomenonTime asc",
        f"https://tld.iot.hamburg.de/v1.1/Datastreams({ID})/Observations?$skip=5000&$top=1000&$orderby=phenomenonTime asc",
        f"https://tld.iot.hamburg.de/v1.1/Datastreams({ID})/Observations?$skip=6000&$top=1000&$orderby=phenomenonTime asc",
        f"https://tld.iot.hamburg.de/v1.1/Datastreams({ID})/Observations?$skip=7000&$top=1000&$orderby=phenomenonTime asc",
        f"https://tld.iot.hamburg.de/v1.1/Datastreams({ID})/Observations?$skip=8000&$top=1000&$orderby=phenomenonTime asc",
        f"https://tld.iot.hamburg.de/v1.1/Datastreams({ID})/Observations?$skip=9000&$top=1000&$orderby=phenomenonTime asc",
    ]

    # Run requests in parallel
    with ThreadPoolExecutor() as executor:
        results = list(executor.map(fetch_data, urls))

    # Combine results into a single list
    combined_results = list()
    for result in results:
        if isinstance(result, dict) and "value" in result:
            combined_results.extend(result["value"])

    observations = [(observation["phenomenonTime"],observation["result"]) for observation in combined_results]

    combined_observations[ID] = observations
    end = time.time()
    counter += 1
    print(f"count: {start_idx+counter}, ID: {ID}, fetch time: {end-start}")
    time.sleep(1)
    if counter%1 == 0:
        try:
            path = f"{os.getcwd()}"
            name = f"observations_ids{start_idx+counter-9}-{start_idx+counter}.json"
            file_path = os.path.join(path,name)
            with open(file_path,"w") as file:
                json.dump(combined_observations, file, indent=4)
            print(f"Data successfully saved to {file}")
            combined_observations = dict()
        except Exception as e:
            print(f"An error occurred while saving to file: {e}")

count: 20021, ID: 91099, fetch time: 1.2759480476379395
Data successfully saved to <_io.TextIOWrapper name='/Users/mikko/OneDrive/Studium/WS24_25/Projekt/temp/primary_signal_observations/observations_ids20012-20021.json' mode='w' encoding='UTF-8'>


In [11]:
try:
    combined_data = dict()
    path = f"{os.getcwd()}"
    for i in range(2003):
        if i%100 ==0:
            print(i)
        name = f"observations_ids{10*i+1}-{10*(i+1)}.json"
        file_path = os.path.join(path,name)
        with open(file_path, "r") as input_file:
            data = json.load(input_file)
            combined_data.update(data)
    name = f"observations_ids20021.json"
    file_path = os.path.join(path,name)
    with open(file_path, "r") as input_file:
        data = json.load(input_file)
        combined_data.update(data)

    name = f"observations_all_datastreams.json"
    file_path = os.path.join(path,name)
    with open(file_path,"w") as combined_file:
        json.dump(combined_data,combined_file,indent=4)
except Exception as e:
    print(f"An error occurred while saving to file: {e}")

0
100
200
300
400
500
600
700
800
900
1000
1100
1200
1300
1400
1500
1600
1700
1800
1900
2000
An error occurred while saving to file: [Errno 2] No such file or directory: '/Users/mikko/OneDrive/Studium/WS24_25/Projekt/temp/primary_signal_observations/observations_ids20021-20030.json'


In [16]:
try:
    name = f"observations_ids20021.json"
    file_path = os.path.join(path,name)
    with open(file_path, "r") as input_file:
        data = json.load(input_file)
        combined_data.update(data)

    name = f"observations_all_datastreams.json"
    file_path = os.path.join(path,name)
    with open(file_path,"w") as combined_file:
        print("saving...")
        json.dump(combined_data,combined_file,indent=4)
        print("saved!")
except Exception as e:
    print(f"An error occurred while saving to file: {e}")

saving...
saved!


In [17]:
sum_green = timedelta(0)
sum_red = timedelta(0)

last_green_observation_time = None
last_red_observation_time = None

green_percentages = dict()
for key,value in combined_data.items():
    for observation in value:
        if observation[1] == 1:
            last_red_observation_time = observation[0]
            if not last_green_observation_time:
                continue
            time_difference = datetime.strptime(last_red_observation_time,"%Y-%m-%dT%H:%M:%SZ") - datetime.strptime(last_green_observation_time,"%Y-%m-%dT%H:%M:%SZ")
            if time_difference < timedelta(minutes=5):
                sum_green += time_difference
        if observation[1] == 3 or observation[1] == 0:
            last_green_observation_time = observation[0]
            if not last_red_observation_time:
                continue
            time_difference = datetime.strptime(last_green_observation_time,"%Y-%m-%dT%H:%M:%SZ") - datetime.strptime(last_red_observation_time,"%Y-%m-%dT%H:%M:%SZ")
            if time_difference < timedelta(minutes=5):
                    sum_red += time_difference
    
    green_percentage = sum_green / (sum_green+sum_red)
    print(key)
    green_percentages[key] = green_percentage


15
6
19
27
18
25
14
11874
4492
7489
1417
888
1331
4475
4090
11901
2958
4687
4685
4539
7645
4639
1191
14838
4580
11868
160
11883
11897
15206
4553
3161
4500
4549
1838
4873
4829
1791
12424
12436
12441
2866
495
1578
4509
11037
12371
1267
3053
1894
3077
3095
12361
998
12338
12342
3669
12353
12347
1500
5701
3744
15474
1670
12375
84
1698
1564
3236
5137
5584
2718
1398
14816
4833
4938
1237
14885
3211
1364
5606
9076
2997
664
1347
3228
10776
2956
721
1421
15341
124
3548
10822
673
14798
1340
15321
559
1325
15221
10865
10798
14892
10804
10810
10782
10833
49
8773
3247
3230
3150
1647
1441
3936
15148
1040
1658
1923
1259
5034
1430
3143
15070
15076
2733
414
46
2708
1683
1675
3249
1843
6513
8783
10828
1907
3169
6580
15250
1307
12691
2610
10816
15556
6057
1941
1909
12611
5876
1851
12669
12661
3182
1944
10829
3369
1328
15482
15490
1954
12677
15363
15257
1310
12699
1240
0
3423
3417
89
1690
9418
5049
3422
2700
1809
3263
1635
15305
5871
11657
5899
5957
10936
7757
380
11707
6741
4854
11674
6562
1645
3397
1680


In [20]:
print(green_percentages)

KeyError: 15